In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import date, datetime, timedelta
from pyspark.sql.window import Window

#Função para traduzir os metadados das tabelas
def traduzir_colunas(df: DataFrame, mapa_colunas: dict) -> DataFrame:
    #Traduz as colunas do dataframe.
    for origem, destino in mapa_colunas.items():
        df = df.withColumnRenamed(origem, destino)
    return df



In [0]:


df_bronze_movies_info = spark.table("workspace.bronze.tb_movies_info")

#Normalizando os status
df_silver_info_filmes = df_bronze_movies_info.withColumn(
    "status",
    F.regexp_replace(F.lower(F.col("status")), "[^a-z]", "")
)

# Traduzindo os status
df_silver_info_filmes = df_silver_info_filmes.withColumn(
    "status",
    F.when(F.col("status") == "released",       "Lançado")
     .when(F.col("status") == "postproduction", "Pós-Produção")
     .when(F.col("status") == "inproduction",   "Em Produção")
     .when(F.col("status") == "planned",        "Planejado")
     .otherwise("Não Informado")
)

# display(df_silver_info_filmes)

In [0]:
mapa_colunas = {
    "id": "id_filme",
    "title": "titulo",
    "original_title": "titulo_original",
    "release_date": "data_lancamento",
    "runtime": "duracao_minutos",
    "original_language": "idioma_original",
    "status": "status_filme",
    "overview": "sinopse",
    "tagline": "frase_divulgacao",
}


df_silver_info_filmes = traduzir_colunas(df_silver_info_filmes, mapa_colunas)
# display(df_silver_info_filmes.limit(10))

In [0]:

#Retirando filmes duplicados por ID
janela = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_silver_info_filmes = (
    df_silver_info_filmes
    .withColumn("rn", F.row_number().over(janela))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
#Debug de qntd de filmes
total = df_silver_info_filmes.count()
distintos = df_silver_info_filmes.select("id_filme").distinct().count()
print(f"Linhas: {total} | Filmes distintos: {distintos}")

Linhas: 97879 | Filmes distintos: 97879


In [0]:

#Descobrindo os padroes de datas
display(
    df_silver_info_filmes
    .select(F.regexp_replace("data_lancamento", r"\d", "9").alias("padrao"))
    .groupBy("padrao").count()
    .orderBy(F.desc("count"))
)

#Formatando as datas
data_txt = F.trim(F.col("data_lancamento"))

df_silver_info_filmes = df_silver_info_filmes.withColumn(
    "data_lancamento",
    F.to_date(
        F.when(data_txt.rlike(r"^\d{4}-\d{2}-\d{2}$"),
               F.expr("try_to_timestamp(trim(data_lancamento), 'yyyy-MM-dd')"))
         .when(data_txt.rlike(r"^\d{2}/\d{2}/\d{4}$"),
               F.expr("try_to_timestamp(trim(data_lancamento), 'dd/MM/yyyy')"))
         .when(data_txt.rlike(r"^\d{2}-\d{2}-\d{4}$"),
               F.expr("try_to_timestamp(trim(data_lancamento), 'dd-MM-yyyy')"))
         .otherwise(None)
    )
)

df_silver_info_filmes = df_silver_info_filmes.withColumn("ano_lancamento", F.year("data_lancamento"))
df_silver_info_filmes.printSchema()

padrao,count
9999-99-99,83816
99/99/9999,9686
99-99-9999,4313
null,56
northern Sweden. The dream was to get back to Nature,1
at least for a brief moment,1
World's greatest detective. World's smallest package.,1
"a boy who delivers drugs from being attacked by a group of gangsters in which Song and Nick are the members.""",1
in partnership with brave rebel Saudi Arabian undercover reporters,1
the more Masuda remembers his younger self,1


root
 |-- id_filme: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)
 |-- ano_lancamento: integer (nullable = true)



In [0]:
df_silver_financeiro_filmes = spark.table("workspace.bronze.tb_movies_financials")

mapa_colunas_2 = {
    "id": "id_filme",
    "budget": "orcamento_usd",
    "revenue": "receita_usd",
}

#Traduzindo os metaDados da tabela info filmes
df_silver_financeiro_filmes = traduzir_colunas(df_silver_financeiro_filmes, mapa_colunas_2)

df_silver_financeiro_filmes.printSchema()

#Tratando os valores ausentes
valores_ausentes = ["unknown", "não informado", "n/a"]

def tratar_ausentes(coluna):
    valor = F.lower(F.trim(F.col(coluna)))
    return F.when(valor.isin(valores_ausentes), None).otherwise(F.trim(F.col(coluna)))

for c in ["orcamento_usd", "receita_usd"]:
    df_silver_financeiro_filmes = df_silver_financeiro_filmes.withColumn(c, tratar_ausentes(c))


def higienizar_valor(c):
    original = F.trim(F.col(c))
    # Remove moeda, vírgulas de milhar e espaços (mantém o sufixo K/M/B)
    sem_moeda = F.regexp_replace(F.upper(original), r"US\$|USD|R\$|\$|,|\s", "")

    # Separa número e sufixo:  "6.0M" → "6.0" e "M"
    padrao = r"^(-?\d+(?:\.\d+)?)([KMB]?)$"
    numero_txt = F.regexp_extract(sem_moeda, padrao, 1)
    sufixo     = F.regexp_extract(sem_moeda, padrao, 2)

    numero = F.when(numero_txt != "", numero_txt).cast("decimal(20,2)")
    multiplicador = (
        F.when(sufixo == "K", 1_000)
         .when(sufixo == "M", 1_000_000)
         .when(sufixo == "B", 1_000_000_000)
         .otherwise(1)
    )
    valor = (numero * multiplicador).cast("decimal(18,2)")
    # Tratando valores zerados ou negativos
    return (
        F.when(valor <= 0, None)
         .otherwise(valor)
    )

for c in ["orcamento_usd", "receita_usd"]:
    df_silver_financeiro_filmes = df_silver_financeiro_filmes.withColumn(c, higienizar_valor(c))

# Janela agrupa todas as versões (cargas) de cada filme
janela_filme = (
    Window.partitionBy("id_filme")
    .orderBy("ingestion_datetime")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

df_silver_financeiro_filmes = (
    df_silver_financeiro_filmes
    .select(
        "id_filme",
        # último orçamento/receita NÃO nulo entre todas as cargas do filme (ou seja, o mais recente disponível)
        F.last("orcamento_usd", ignorenulls=True).over(janela_filme).alias("orcamento_usd"),
        F.last("receita_usd",   ignorenulls=True).over(janela_filme).alias("receita_usd"),
    )
    .dropDuplicates(["id_filme"])
)


root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: string (nullable = true)
 |-- receita_usd: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



orcamento_usd,receita_usd
null,null
null,null
null,null
5000000.00,null
null,null


In [0]:
#devolve a cotaçao com o maior dataHoraCotacao ou seja a mais recente
ultima = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.desc("dataHoraCotacao"))
    .first()
)

cotacao = ultima["cotacaoCompra"]
dataCotacao = ultima["dataHoraCotacao"]

print(f"Cotação utilizada: R$ {cotacao} ({dataCotacao})")

df_silver_financeiro_filmes = (
    df_silver_financeiro_filmes
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    .withColumn("receita_brl",   F.round(F.col("receita_usd")   * F.lit(cotacao), 2).cast("decimal(18,2)"))
)

display(
    df_silver_financeiro_filmes
    .select("id_filme", "orcamento_usd", "orcamento_brl", "receita_usd", "receita_brl")
    .limit(10)
)



Cotação utilizada: R$ 5.082 (2026-10-16 13:04:14.548156)


id_filme,orcamento_usd,orcamento_brl,receita_usd,receita_brl
1000004,null,null,null,null
1000005,null,null,null,null
1000030,null,null,null,null
1000081,5000000.00,25410000.00,null,null
1000092,null,null,null,null
1000094,null,null,null,null
1000096,null,null,null,null
1000108,null,null,null,null
1000194,null,null,null,null
1000197,null,null,null,null


In [0]:
#Trocando o nulo por zero para nao interferir na conta do lucro
receita_usd   = F.coalesce(F.col("receita_usd"),   F.lit(0))
orcamento_usd = F.coalesce(F.col("orcamento_usd"), F.lit(0))
receita_brl   = F.coalesce(F.col("receita_brl"),   F.lit(0))
orcamento_brl = F.coalesce(F.col("orcamento_brl"), F.lit(0))

# Fazendo contas dos lucros e margens
df_silver_financeiro_filmes = (
    df_silver_financeiro_filmes
    .withColumn("lucro_usd", (receita_usd - orcamento_usd).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (receita_brl - orcamento_brl).cast("decimal(18,2)"))
    .withColumn(
        "margem_lucro_%",
        F.when(receita_usd > 0, F.round(F.col("lucro_usd") / receita_usd * 100, 2))
         .otherwise(None)
         .cast("decimal(10,2)")
    )
    
)
display(
    df_silver_financeiro_filmes.select(
        "id_filme", "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_brl", "receita_brl", "lucro_brl", "margem_lucro_%"
    )
    .limit(5)
)
linhas = df_silver_financeiro_filmes.count()
filmes = df_silver_financeiro_filmes.select("id_filme").distinct().count()
print(f"Linhas: {linhas} | Filmes: {filmes}")

id_filme,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,margem_lucro_%
1000004,null,null,0.00,null,null,0.00,null
1000005,null,null,0.00,null,null,0.00,null
1000030,null,null,0.00,null,null,0.00,null
1000081,5000000.00,null,-5000000.00,25410000.00,null,-25410000.00,null
1000092,null,null,0.00,null,null,0.00,null


Linhas: 99006 | Filmes: 99006


In [0]:
df_silver_metricas_engajamento = spark.table("workspace.bronze.tb_movies_metrics")

mapa_colunas_3 = {
    "id":            "id_filme",
    "popularity":    "popularidade",
    "vote_average":  "nota_media_tmdb",
    "vote_count":    "qtd_votos_tmdb",
    "averageRating": "nota_media_imdb",
    "numVotes":      "qtd_votos_imdb",
}

df_silver_metricas_engajamento=traduzir_colunas(df_silver_metricas_engajamento, mapa_colunas_3)

df_silver_metricas_engajamento.printSchema()


root
 |-- id_filme: string (nullable = true)
 |-- popularidade: string (nullable = true)
 |-- nota_media_tmdb: string (nullable = true)
 |-- qtd_votos_tmdb: string (nullable = true)
 |-- nota_media_imdb: string (nullable = true)
 |-- qtd_votos_imdb: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
def limpar_decimal(c):
    # mantém só dígitos, ponto, vírgula e sinal
    txt = F.regexp_replace(F.trim(F.col(c)), r"[^0-9.,\-]", "")

    tem_ponto   = txt.contains(".")
    tem_virgula = txt.contains(",")
    qtd_pontos   = F.size(F.split(txt, r"\.")) - 1
    qtd_virgulas = F.size(F.split(txt, ",")) - 1
    virgula_por_ultimo = txt.rlike(r",\d*$")

    limpo = (
        # ponto e vírgula juntos: o último é o decimal
        F.when(tem_ponto & tem_virgula & virgula_por_ultimo,
               F.regexp_replace(F.regexp_replace(txt, r"\.", ""), ",", "."))   # 1.234,56 → 1234.56
         .when(tem_ponto & tem_virgula,
               F.regexp_replace(txt, ",", ""))                                  # 1,234.56 → 1234.56
        # só vírgula
         .when(tem_virgula & (qtd_virgulas > 1), F.regexp_replace(txt, ",", ""))  # 1,234,567
         .when(tem_virgula, F.regexp_replace(txt, ",", "."))                     # 12,345 → 12.345
        # só ponto
         .when(qtd_pontos > 1, F.regexp_replace(txt, r"\.", ""))                 # 1.234.567
         .otherwise(txt)                                                         # 12.345 ou 12
    )

    # só converte se o resultado for um número válido
    return F.when(limpo.rlike(r"^-?\d+(\.\d+)?$"), limpo)


df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .withColumn("popularidade", limpar_decimal("popularidade"))
)

display(df_silver_metricas_engajamento.select("popularidade"))

popularidade
72.735
154.34
91.756
54.522
70.741
43.665
70.535
65.88
67.553
35.356


In [0]:
colunas_nota = ["nota_media_tmdb", "nota_media_imdb"]   # decimais
colunas_qtd  = ["qtd_votos_tmdb", "qtd_votos_imdb"]     # inteiros

# O try_cast converte o valor  e se nao conseguir  devolve NULL
#  Converte para NULL se nao tiver entre 0 e 10 ou senao for numero
for c in colunas_nota:
    df_silver_metricas_engajamento = df_silver_metricas_engajamento.withColumn(
        c, F.expr(f"try_cast(trim({c}) AS DECIMAL(4,2))")
    )
    df_silver_metricas_engajamento = df_silver_metricas_engajamento.withColumn(
        c, F.when(F.col(c).between(0, 10), F.col(c))
    )


# Converte para NULL se for <0 ou nao for numero
for c in colunas_qtd:
    df_silver_metricas_engajamento = df_silver_metricas_engajamento.withColumn(
        c, F.expr(f"try_cast(trim({c}) AS BIGINT)")
    )
    df_silver_metricas_engajamento = df_silver_metricas_engajamento.withColumn(
        c, F.when(F.col(c) >= 0, F.col(c))
    )

# Janela agrupa todas as versões (cargas) de cada filme
janela_filme = (
    Window.partitionBy("id_filme")
    .orderBy("ingestion_datetime")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

colunas_metricas = ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]

df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .select(
        "id_filme",
        # último de cada metrica NÃO nulo entre todas as cargas do filme (ou seja, o mais recente disponível)
        *[F.last(c, ignorenulls=True).over(janela_filme).alias(c) for c in colunas_metricas],
    )
    .dropDuplicates(["id_filme"])
)

print("Depois da consolidação:", df_silver_metricas_engajamento.count(), "linhas")

linhas = df_silver_metricas_engajamento.count()
filmes = df_silver_metricas_engajamento.select("id_filme").distinct().count()
print(f"Linhas: {linhas} | Filmes: {filmes}")


Depois da consolidação: 99013 linhas
Linhas: 99013 | Filmes: 99013


In [0]:
df_silver_avaliacoes_usuarios = spark.table("workspace.bronze.tb_movies_reviews")

mapa_colunas_4 = {
    "id":         "id_filme",
    "nome":       "nome_usuario",
    "nota":       "nota_usuario",
    "comentario": "comentario_usuario",
}

df_silver_avaliacoes_usuarios=traduzir_colunas(df_silver_avaliacoes_usuarios,mapa_colunas_4)
df_silver_avaliacoes_usuarios.printSchema()

# Remove duplicatas
colunas_chave = ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]

df_silver_avaliacoes_usuarios = df_silver_avaliacoes_usuarios.dropDuplicates(colunas_chave)

root
 |-- id_filme: string (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: string (nullable = true)
 |-- comentario_usuario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:

# O try_cast converte o valor  e se nao conseguir  devolve NULL
#  Converte para NULL se nao tiver entre 0 e 10 ou senao for numero
df_silver_avaliacoes_usuarios = df_silver_avaliacoes_usuarios.withColumn(
    "nota_usuario", F.expr(f"try_cast(trim({"nota_usuario"}) AS DECIMAL(4,1))")
)
df_silver_avaliacoes_usuarios = df_silver_avaliacoes_usuarios.withColumn(
    "nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario"))
)
display(df_silver_avaliacoes_usuarios.select("nota_usuario").limit(5))

nota_usuario
1.1
7.2
9.7
7.7
8.4


In [0]:
#Retira os espaços em branco do comentario
comentario = F.trim(F.col("comentario_usuario"))
#Se o comentario for nulo ou vazio, substitui por "Sem comentário"
df_silver_avaliacoes_usuarios = df_silver_avaliacoes_usuarios.withColumn(
    "comentario_usuario",
    F.when(comentario.isNull() | (comentario == ""), F.lit("Sem comentário"))
     .otherwise(comentario)
)

In [0]:
df_silver_credits_and_tag= spark.table("workspace.bronze.tb_credits_and_tags")
df_silver_credits_and_tag = (df_silver_credits_and_tag
              .withColumn("genres", F.regexp_replace(F.col("genres"), ";", ",")) # Substitui ; por ,
              .withColumn("genres", F.regexp_replace(F.col("genres"), r"[;|]", ",")) # Substitui | por ,
              )
df_silver_credits_and_tag =(df_silver_credits_and_tag
             .withColumn("genres", F.explode(F.split(F.col("genres"), ","))) #split + explode
             .withColumn("genres", F.trim(F.col("genres"))) #retira os espaços em branco
             .filter(F.col("genres") != "") #retira os generos vazios
             )

generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]

df_silver_generos = (
    df_silver_credits_and_tag
    .select(F.col("id").alias("id_filme"), F.col("genres").alias("genero"))
    .withColumn("genero", F.regexp_replace(F.col("genero"), r'["\\/]+', ","))   # aspas e barras viram ,
    .withColumn("genero", F.explode(F.split(F.col("genero"), ",")))             # separa o que estava grudado
    .withColumn("genero", F.initcap(F.trim(F.col("genero"))))                   # tira espaços e padroniza a caixa
    .withColumn("genero", F.regexp_replace(F.col("genero"), "^Tv Movie$", "TV Movie"))
    .filter(F.col("genero").isin(generos_validos))                              # só gêneros oficiais
    .dropDuplicates(["id_filme", "genero"])
)

print("Gêneros distintos:", df_silver_generos.select("genero").distinct().count())
display(df_silver_generos.groupBy("genero").count().orderBy(F.desc("count")))



display(df_silver_generos.limit(15))

Gêneros distintos: 19


genero,count
Drama,32687
Documentary,19255
Comedy,18865
Thriller,10415
Horror,9871
Romance,7724
Action,6114
Crime,4795
Animation,4532
TV Movie,4122


id_filme,genero
343668,Comedy
502356,Animation
291870,Comedy
547016,Fantasy
398181,Drama
379170,TV Movie
579047,Science Fiction
804150,Crime
504253,Animation
664469,History


In [0]:
mapa_colunas_5 = {
    "id":                   "id_filme",
    "genres":               "generos",
    "production_companies": "produtoras",
    "production_countries": "paises_producao",
    "spoken_languages":     "idiomas_falados",
    "keywords":             "palavras_chave",
    "directors":            "diretores",
    "writers":              "roteiristas",
    "cast":                 "elenco",
}
#Traduz os Metadados
df_silver_credits_and_tag = traduzir_colunas(df_silver_credits_and_tag,mapa_colunas_5)
df_silver_credits_and_tag.printSchema()

root
 |-- id_filme: string (nullable = true)
 |-- generos: string (nullable = false)
 |-- produtoras: string (nullable = true)
 |-- paises_producao: string (nullable = true)
 |-- idiomas_falados: string (nullable = true)
 |-- palavras_chave: string (nullable = true)
 |-- diretores: string (nullable = true)
 |-- roteiristas: string (nullable = true)
 |-- elenco: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
colunas_pessoas=["elenco","diretores","roteiristas", "produtoras"]
for c in colunas_pessoas:
    df_silver_credits_and_tag = (df_silver_credits_and_tag
                .withColumn(c, F.regexp_replace(F.col(c), ";", ",")) # Substitui ; por ,
                .withColumn(c, F.regexp_replace(F.col(c), r"[;|]", ",")) # Substitui | por ,
                )
    
mapa_tipos = {
    "elenco":      "Ator",
    "diretores":   "Diretor",
    "roteiristas": "Roteirista",
    "produtoras":  "Produtora",
}

partes = []
for coluna, tipo in mapa_tipos.items():
    parte = (
        df_silver_credits_and_tag
        .select(
            F.col("id_filme").cast("string").alias("id_filme"),
            F.explode(F.split(F.regexp_replace(F.col(coluna), r'["\\;|/]+', ","), ",")).alias("nome_entidade"),
        )
        .withColumn("tipo_entidade", F.lit(tipo))
    )
    partes.append(parte)

df_silver_pessoas_empresas = partes[0]
for parte in partes[1:]:
    df_silver_pessoas_empresas = df_silver_pessoas_empresas.unionByName(parte)

# filtro de formato: só o que parece um nome
nome = F.trim(F.col("nome_entidade"))
parece_nome = (
    nome.rlike(r"^[\p{L}][\p{L}0-9 .'&\-]*$")    # começa com letra; só letras, espaço, ponto, apóstrofo, & hífen
    & (F.length(nome) <= 60)                     # sinopses são longas
    & (F.size(F.split(nome, " ")) <= 5)          # nomes têm poucas palavras
)

df_silver_pessoas_empresas = (
    df_silver_pessoas_empresas
    .withColumn("nome_entidade", nome)
    .filter(parece_nome)
    .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))   # tom HANKS → Tom Hanks
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

com_lixo = df_silver_pessoas_empresas.filter(F.col("nome_entidade").rlike(r'["\\]'))
print("Nomes com aspas ou barras:", com_lixo.count())
display(com_lixo.limit(50))
display(df_silver_pessoas_empresas.limit(5))

id_filme,nome_entidade,tipo_entidade
339403,Ansel Elgort,Ator
429617,Tony Revolori,Ator
335984,Dave Bautista,Ator
246655,Evan Peters,Ator
508442,Phylicia Rashād,Ator


In [0]:
df_bronze_cotacao = spark.table(f"workspace.bronze.tb_cotacao_dolar")

#Tabela com cotacoes sem todos os dias
df_cotacao = (
    df_bronze_cotacao
    .withColumn("dataHoraCotacao", F.to_timestamp("dataHoraCotacao"))
    .withColumn("dataCotacao", F.to_date("dataHoraCotacao"))
    .groupBy("dataCotacao")
    .agg(F.max_by("cotacaoCompra", "dataHoraCotacao").alias("cotacaoCompra"))
)


# Cria tabela de dias totais
df_calendario = (
    df_cotacao
    .agg(F.min("dataCotacao").alias("inicio"), F.max("dataCotacao").alias("fim"))
    .select(F.explode(F.sequence("inicio", "fim", F.expr("interval 1 day"))).alias("dataCotacao"))
)


janela = Window.orderBy("dataCotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao = (
    df_calendario
    .join(df_cotacao, on="dataCotacao", how="left") # dias sem cotação ficam NULL
    .withColumn("cotacaoCompra", F.last("cotacaoCompra", ignorenulls=True).over(janela))  # forward fill
    .withColumn("cotacaoCompra", F.col("cotacaoCompra").cast("decimal(10,4)"))
    .orderBy("dataCotacao")
)

display(df_silver_cotacao)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dataCotacao,cotacaoCompra
2026-10-09,4.9886
2026-10-10,4.9886
2026-10-11,4.9886
2026-10-12,4.9886
2026-10-13,5.0090
2026-10-14,5.0435
2026-10-15,5.0515
2026-10-16,5.0820


In [0]:
SILVER_BD = "silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{SILVER_BD}")

tabelas_silver = {
    "tb_info_filmes":          df_silver_info_filmes,
    "tb_financeiro_filmes":    df_silver_financeiro_filmes,
    "tb_metricas_engajamento": df_silver_metricas_engajamento,
    "tb_avaliacoes_usuarios":  df_silver_avaliacoes_usuarios,
    "tb_generos":              df_silver_generos,
    "tb_pessoas_empresas":     df_silver_pessoas_empresas,
    "tb_cotacao_dolar":        df_silver_cotacao,
}

for nome_tabela, df in tabelas_silver.items():
    destino = f"workspace.{SILVER_BD}.{nome_tabela}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    print(f"✔ {destino}: {spark.table(destino).count()} linhas")

✔ workspace.silver.tb_info_filmes: 97879 linhas
✔ workspace.silver.tb_financeiro_filmes: 99006 linhas
✔ workspace.silver.tb_metricas_engajamento: 99013 linhas
✔ workspace.silver.tb_avaliacoes_usuarios: 32412 linhas
✔ workspace.silver.tb_generos: 142188 linhas
✔ workspace.silver.tb_pessoas_empresas: 842047 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✔ workspace.silver.tb_cotacao_dolar: 8 linhas
